In [ ]:
#-- load libraries
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(forcats)
library(paletteer)
library(ggalluvial)
library(scales)

In [ ]:
#-- Read in outcome data
final <- read_delim("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_Phenoconversion_noAD_2scripts_S_C.csv", delim = ",", show_col_types = FALSE)

In [ ]:
#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")

#-- Join outcomes with phenotypes
library(lubridate)
both <- final %>%
  left_join(pheno, by = "person_id") %>%
  mutate(
    age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
  )

In [ ]:
#-- Pooled switching case and control counts
switch <- final %>%
    filter(FinalClassification.treatment == "Switching")
case <- length(unique(switch$person_id))
total <- length(unique(final$person_id))
case; total; total-case

In [ ]:
#-- Summarise the number of AD scripts per person
#-- Read into R the cleaned prescription data 
PrescriptionData <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/AD_processed.csv") %>%
  mutate(drug_exposure_start_date = as.Date(drug_exposure_start_date),
         final_end_date = as.Date(final_end_date))

#-- Filter for Antidepressants
PrescriptionData_AD <- PrescriptionData %>%
  filter(!Category %in% c("Atypical Antipsychotic", "Mood_Stabilizer", "Typical Antipsychotic", 
                          "Serotonin_Precursor", "AminoAcid", "Augmentation"))
dim(PrescriptionData_AD)
length(unique(PrescriptionData_AD$person_id))

In [ ]:
#-- Filter prescription data for those in our final analytical sample
incl_pres <- PrescriptionData_AD %>%
    filter(person_id %in% final$person_id)
dim(incl_pres)

In [ ]:
num_pres <- incl_pres %>%
    group_by(person_id) %>%
    summarise(
    total = n())

In [ ]:
num_pres_plot <- num_pres %>%
  mutate(total_capped = ifelse(total > 250, 251, total))

px <- ggplot(num_pres_plot, aes(x = total_capped)) +
  geom_histogram(binwidth = 10, fill = "steelblue", color = "white") +
  scale_x_continuous(
    breaks = c(0, 50, 100, 150, 200, 250),
    labels = c("0", "50", "100", "150", "200", "250+")
  ) +
  labs(x = "Total AD prescriptions", y = "Number of people in analytical sample") +
  theme_classic()

num_pres_plot_zoom <- num_pres %>%
  filter(total <= 10)

pxx <- ggplot(num_pres_plot_zoom, aes(x = total)) +
  geom_histogram(binwidth = 1, fill = "steelblue", color = "white") +
  scale_x_continuous(
    breaks = c(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
    labels = c("0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10")
  ) +
  labs(x = "Total AD prescriptions", y = "Number of people in analytical sample") +
  theme_classic()

library(cowplot)
all <- plot_grid(px, pxx, nrow = 1, labels = c("A", "B"))
all

In [ ]:
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Num_Prescriptions.png",
  plot     = all,
  width    = 7,
  height   = 5,
  dpi      = 600
)

In [ ]:
#==== Summarise group counts ===============================================================
final = final %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

order <- final %>%
  group_by(drug.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  as.data.frame() %>%
  arrange(desc(NumPeople)) %>%
  slice_head(n = 15)

order <- order$drug.treatment

group_counts <- final %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  mutate(FinalClassification.treatment = ifelse(FinalClassification.treatment == "Switching", "Switching", "Non-Switching")) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Non-Switching"))


colors <- c(
    "Non-Switching" = "#0D47A1FF",
    "Switching" = "#F57F17FF")

p1 <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Antidepressant") +
  ylab("Count") +
  #labs(title = "AoU: Full Prescribing Cohort") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),  # x, y coordinates (0-1 scale)
    legend.justification = c(1, 1),   # anchor point (right, top)
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 30000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))  # 10K, 20K, etc.
p1

In [ ]:
#==== Summarise group counts within MDD ===============================================================
mdd = both %>%
    filter(Depression == 1) %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

order <- mdd %>%
  group_by(drug.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  as.data.frame() %>%
  arrange(desc(NumPeople)) %>%
  slice_head(n = 15)

order <- order$drug.treatment

group_counts <- mdd %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  mutate(FinalClassification.treatment = ifelse(FinalClassification.treatment == "Switching", "Switching", "Non-Switching")) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Non-Switching"))


colors <- c(
    "Non-Switching" = "#0D47A1FF",
    "Switching" = "#F57F17FF")

p1b <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Antidepressant") +
  ylab("Count") +
  labs(fill = "Antidepressant\nOutcome", title = "AoU: With Depression Diagnosis") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),  # x, y coordinates (0-1 scale)
    legend.justification = c(1, 1),   # anchor point (right, top)
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 30000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))  # 10K, 20K, etc.
p1b

In [ ]:
#==== Summarise group counts within noMDD ===============================================================
nmdd = both %>%
    filter(Depression == 0) %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

order <- nmdd %>%
  group_by(drug.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  as.data.frame() %>%
  arrange(desc(NumPeople)) %>%
  slice_head(n = 15)

order <- order$drug.treatment

group_counts <- nmdd %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  mutate(FinalClassification.treatment = ifelse(FinalClassification.treatment == "Switching", "Switching", "Non-Switching")) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Non-Switching"))


colors <- c(
    "Non-Switching" = "#0D47A1FF",
    "Switching" = "#F57F17FF")

p1c <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Antidepressant") +
  ylab("Count") +
  labs(fill = "Antidepressant\nOutcome", title = "AoU: Without Depression Diagnosis") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),  # x, y coordinates (0-1 scale)
    legend.justification = c(1, 1),   # anchor point (right, top)
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 30000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))  # 10K, 20K, etc.
p1c

In [ ]:
library(cowplot)

first <- plot_grid(p1, p1b, p1c, labels = c("A", "B", "C"), ncol = 1)
first

In [ ]:
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Common_ADs.png",
  plot     = first,
  width    = 7,
  height   = 12,
  dpi      = 600
)

In [ ]:
first_switch <- final %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  filter(Incidence == min(Incidence)) %>%
  filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
  ungroup() %>%
  mutate(Time2FirstSwitch = difftime(Incidence, ExpectedEndDate.treatment.episode, units = "days")) %>%
  mutate(TimeFirstDrug = difftime(ExpectedEndDate.treatment.episode, FirstDispense.treatment.episode, unit = "days")) %>%
  mutate(TimeSecondDrug = difftime(ExpectedEndDate.other.episode, FirstDispense.other.episode, unit = "days"))

cross_taper <- first_switch %>%
    filter(Time2FirstSwitch < 0)
dim(cross_taper)
dim(first_switch)

switching_summary <- first_switch %>%
  summarise(
    Count = n(),
    
    # Time to first switch
    mean_t2s = mean(Time2FirstSwitch, na.rm = TRUE),
    median_t2s = median(Time2FirstSwitch, na.rm = TRUE),
    q1_t2s = quantile(Time2FirstSwitch, 0.25, na.rm = TRUE),
    q3_t2s = quantile(Time2FirstSwitch, 0.75, na.rm = TRUE),
    iqr_t2s = IQR(Time2FirstSwitch, na.rm = TRUE),
    
    # Time on first drug
    mean_tFD = mean(TimeFirstDrug, na.rm = TRUE),
    median_tFD = median(TimeFirstDrug, na.rm = TRUE),
    q1_tFD = quantile(TimeFirstDrug, 0.25, na.rm = TRUE),
    q3_tFD = quantile(TimeFirstDrug, 0.75, na.rm = TRUE),
    iqr_tFD = IQR(TimeFirstDrug, na.rm = TRUE),
    
    # Time on second drug
    mean_tSD = mean(TimeSecondDrug, na.rm = TRUE),
    median_tSD = median(TimeSecondDrug, na.rm = TRUE),
    q1_tSD = quantile(TimeSecondDrug, 0.25, na.rm = TRUE),
    q3_tSD = quantile(TimeSecondDrug, 0.75, na.rm = TRUE),
    iqr_tSD = IQR(TimeSecondDrug, na.rm = TRUE)
  )

In [ ]:
switching_summary

In [ ]:
switch_df <- first_switch %>%
  mutate(first_class = ifelse(!Category.treatment %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.treatment)) %>%
  mutate(second_class = ifelse(!Category.other %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.other)) %>%
  group_by(first_class, second_class) %>%
  summarise(
    freq = n(),
    .groups = "drop"
  )

In [ ]:
p2 <- ggplot(
  switch_df,
  aes(
    axis1 = first_class,
    axis2 = second_class,
    y = freq
  )
) +
  geom_alluvium(aes(fill = first_class),
                width = 0.15,
                alpha = 0.8) +
  geom_stratum(width = 0.15) +
  geom_text(
    stat = "stratum",
    aes(label = after_stat(stratum)),
    size = 3,
    fontface = "bold"
  ) +
  scale_x_discrete(
    limits = c("First\ndrugclass", "Second\ndrugclass"),
    expand = c(0.1, 0.1)
  ) +
  labs(
    x = NULL,
    y = "All of Us Participants (N=34,657)",
    fill = "First class",
    title = NULL
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    #axis.text.y = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.title = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    axis.text.x = element_text(size = 12, face = "bold", color = "black")
  )

In [ ]:
p2

In [ ]:
# Same but within MDD
first_switch <- both %>%
  filter(Depression == 1) %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  filter(Incidence == min(Incidence)) %>%
  filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
  ungroup() %>%
  mutate(Time2FirstSwitch = difftime(Incidence, ExpectedEndDate.treatment.episode, units = "days")) %>%
  mutate(TimeFirstDrug = difftime(ExpectedEndDate.treatment.episode, FirstDispense.treatment.episode, unit = "days")) %>%
  mutate(TimeSecondDrug = difftime(ExpectedEndDate.other.episode, FirstDispense.other.episode, unit = "days"))
dim(first_switch)

cross_taper <- first_switch %>%
    filter(Time2FirstSwitch < 0)
dim(cross_taper)

switching_summary <- first_switch %>%
  summarise(
    Count = n(),
    
    # Time to first switch
    mean_t2s = mean(Time2FirstSwitch, na.rm = TRUE),
    median_t2s = median(Time2FirstSwitch, na.rm = TRUE),
    q1_t2s = quantile(Time2FirstSwitch, 0.25, na.rm = TRUE),
    q3_t2s = quantile(Time2FirstSwitch, 0.75, na.rm = TRUE),
    iqr_t2s = IQR(Time2FirstSwitch, na.rm = TRUE),
    
    # Time on first drug
    mean_tFD = mean(TimeFirstDrug, na.rm = TRUE),
    median_tFD = median(TimeFirstDrug, na.rm = TRUE),
    q1_tFD = quantile(TimeFirstDrug, 0.25, na.rm = TRUE),
    q3_tFD = quantile(TimeFirstDrug, 0.75, na.rm = TRUE),
    iqr_tFD = IQR(TimeFirstDrug, na.rm = TRUE),
    
    # Time on second drug
    mean_tSD = mean(TimeSecondDrug, na.rm = TRUE),
    median_tSD = median(TimeSecondDrug, na.rm = TRUE),
    q1_tSD = quantile(TimeSecondDrug, 0.25, na.rm = TRUE),
    q3_tSD = quantile(TimeSecondDrug, 0.75, na.rm = TRUE),
    iqr_tSD = IQR(TimeSecondDrug, na.rm = TRUE)
  )

In [ ]:
switching_summary

In [ ]:
switch_df <- first_switch %>%
  mutate(first_class = ifelse(!Category.treatment %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.treatment)) %>%
  mutate(second_class = ifelse(!Category.other %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.other)) %>%
  group_by(first_class, second_class) %>%
  summarise(
    freq = n(),
    .groups = "drop"
  )

In [ ]:
p2b <- ggplot(
  switch_df,
  aes(
    axis1 = first_class,
    axis2 = second_class,
    y = freq
  )
) +
  geom_alluvium(aes(fill = first_class),
                width = 0.15,
                alpha = 0.8) +
  geom_stratum(width = 0.15) +
  geom_text(
    stat = "stratum",
    aes(label = after_stat(stratum)),
    size = 3,
    fontface = "bold"
  ) +
  scale_x_discrete(
    limits = c("First\ndrugclass", "Second\ndrugclass"),
    expand = c(0.1, 0.1)
  ) +
  labs(
    x = NULL,
    y = "All of Us Participants (N=24,179)",
    fill = "First class",
    title = NULL
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    #axis.text.y = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.title = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    axis.text.x = element_text(size = 12, face = "bold", color = "black")
  )
p2b

In [ ]:
# Same but without MDD
first_switch <- both %>%
  filter(Depression == 0) %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  filter(Incidence == min(Incidence)) %>%
  filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
  ungroup() %>%
  mutate(Time2FirstSwitch = difftime(Incidence, ExpectedEndDate.treatment.episode, units = "days")) %>%
  mutate(TimeFirstDrug = difftime(ExpectedEndDate.treatment.episode, FirstDispense.treatment.episode, unit = "days")) %>%
  mutate(TimeSecondDrug = difftime(ExpectedEndDate.other.episode, FirstDispense.other.episode, unit = "days"))
dim(first_switch)

cross_taper <- first_switch %>%
    filter(Time2FirstSwitch < 0)
dim(cross_taper)

switching_summary <- first_switch %>%
  summarise(
    Count = n(),
    
    # Time to first switch
    mean_t2s = mean(Time2FirstSwitch, na.rm = TRUE),
    median_t2s = median(Time2FirstSwitch, na.rm = TRUE),
    q1_t2s = quantile(Time2FirstSwitch, 0.25, na.rm = TRUE),
    q3_t2s = quantile(Time2FirstSwitch, 0.75, na.rm = TRUE),
    iqr_t2s = IQR(Time2FirstSwitch, na.rm = TRUE),
    
    # Time on first drug
    mean_tFD = mean(TimeFirstDrug, na.rm = TRUE),
    median_tFD = median(TimeFirstDrug, na.rm = TRUE),
    q1_tFD = quantile(TimeFirstDrug, 0.25, na.rm = TRUE),
    q3_tFD = quantile(TimeFirstDrug, 0.75, na.rm = TRUE),
    iqr_tFD = IQR(TimeFirstDrug, na.rm = TRUE),
    
    # Time on second drug
    mean_tSD = mean(TimeSecondDrug, na.rm = TRUE),
    median_tSD = median(TimeSecondDrug, na.rm = TRUE),
    q1_tSD = quantile(TimeSecondDrug, 0.25, na.rm = TRUE),
    q3_tSD = quantile(TimeSecondDrug, 0.75, na.rm = TRUE),
    iqr_tSD = IQR(TimeSecondDrug, na.rm = TRUE)
  )

switch_df <- first_switch %>%
  mutate(first_class = ifelse(!Category.treatment %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.treatment)) %>%
  mutate(second_class = ifelse(!Category.other %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.other)) %>%
  group_by(first_class, second_class) %>%
  summarise(
    freq = n(),
    .groups = "drop"
  )

In [ ]:
p2c <- ggplot(
  switch_df,
  aes(
    axis1 = first_class,
    axis2 = second_class,
    y = freq
  )
) +
  geom_alluvium(aes(fill = first_class),
                width = 0.15,
                alpha = 0.8) +
  geom_stratum(width = 0.15) +
  geom_text(
    stat = "stratum",
    aes(label = after_stat(stratum)),
    size = 3,
    fontface = "bold"
  ) +
  scale_x_discrete(
    limits = c("First\ndrugclass", "Second\ndrugclass"),
    expand = c(0.1, 0.1)
  ) +
  labs(
    x = NULL,
    y = "All of Us Participants (N=24,179)",
    fill = "First class",
    title = NULL
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    #axis.text.y = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.title = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    axis.text.x = element_text(size = 12, face = "bold", color = "black")
  )
p2c

In [ ]:
#-- ggalluvial plot for LL
library(readxl)
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table5")
sum(dat$freq)

p2b <- ggplot(
  dat,
  aes(
    axis1 = first_class,
    axis2 = second_class,
    y = freq
  )
) +
  geom_alluvium(aes(fill = first_class),
                width = 0.15,
                alpha = 0.8) +
  geom_stratum(width = 0.15) +
  geom_text(
    stat = "stratum",
    aes(label = after_stat(stratum)),
    size = 3,
    fontface = "bold"
  ) +
  scale_x_discrete(
    limits = c("First\ndrugclass", "Second\ndrugclass"),
    expand = c(0.1, 0.1)
  ) +
  labs(
    x = NULL,
    y = "Pharmlines Participants (N=2,192)",
    fill = "First class",
    title = NULL
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    #axis.text.y = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.title = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    axis.text.x = element_text(size = 12, face = "bold", color = "black")
  ) +
  scale_fill_manual(
    values = c(
      "SSRI" = "#A58AFF",
      "TCA" = "#FB61D7",
      "SNRI" = "#00B6EB",
      "NaSSA" = "#F87660",
      "Other" = "#53B400"
    )
  )
p2b

In [ ]:
#-- Summaries for the top 15 drugs
drug_switch <- first_switch %>%
    mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment))

p3 <- ggplot(drug_switch, aes(x=drug.treatment, y = Time2FirstSwitch, fill = drug.treatment)) +
    geom_boxplot(outlier.shape = NA) +
    theme_classic() +
    labs(x = "Index Antidepressant", y = "Time to first switch\nafter termination of index drug (days)", fill = "First Class") +
    theme(
    axis.title = element_text(face = "bold"),
    axis.text = element_text(color = "black"),
    axis.text.x = element_text(angle = 45, hjust=1),
    legend.position = "none")   +
    paletteer::scale_fill_paletteer_d("khroma::iridescent", direction =-1)
p3

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Time_2_Switch.png",
  plot     = p3,
  width    = 8,
  height   = 7,
  dpi      = 600
)


In [ ]:
#-- First switch
first_switch <- both %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- First continuation (among non-switchers)
C <- both %>%
  filter(!person_id %in% first_switch$person_id) %>%
  filter(FinalClassification.treatment == "Continuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- Input data for switching vs. non-switching
dat <- rbind(first_switch, C)


In [ ]:
full <- dat %>%
    group_by(FinalClassification.treatment) %>%
    summarise(n = n()) %>%
    mutate(Subset = "Full")
mdd <- dat %>%
    filter(Depression == 1) %>%
    group_by(FinalClassification.treatment) %>%
    summarise(n = n()) %>%
    mutate(Subset = "MDD")
nomdd <- dat %>%
    filter(Depression == 0) %>%
    group_by(FinalClassification.treatment) %>%
    summarise(n = n()) %>%
    mutate(Subset = "noMDD")

all <- rbind(full, mdd, nomdd)

colors <- c(
    "Continuation" = "#50C878",
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF")

p4 <- ggplot(all, aes(x = FinalClassification.treatment, y = n, fill = FinalClassification.treatment)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Antidepressant") +
  ylab("All of Us Participant Count") +
  labs(fill = "Antidepressant\nOutcome") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),  # x, y coordinates (0-1 scale)
    legend.justification = c(1, 1),   # anchor point (right, top)
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold"),
      strip.text = element_text(face = "bold")
  ) +
 facet_wrap(Subset ~ .) +
  scale_y_continuous(
    breaks = seq(0, 70000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))  # 10K, 20K, etc.
p4

In [ ]:
full <- dat %>%
    filter(!Category.treatment %in% c("NRI", "MAOI", "NMDA_Antagonist", "Serotonin_Modulator")) %>%
    group_by(Category.treatment, FinalClassification.treatment) %>%
    summarise(n = n()) %>%
    mutate(Subset = "Full")
mdd <- dat %>%
    filter(!Category.treatment %in% c("NRI", "MAOI", "NMDA_Antagonist", "Serotonin_Modulator")) %>%
    filter(Depression == 1) %>%
    group_by(Category.treatment, FinalClassification.treatment) %>%
    summarise(n = n()) %>%
    mutate(Subset = "MDD")
nomdd <- dat %>%
    filter(!Category.treatment %in% c("NRI", "MAOI", "NMDA_Antagonist", "Serotonin_Modulator")) %>%
    filter(Depression == 0) %>%
    group_by(Category.treatment, FinalClassification.treatment) %>%
    summarise(n = n()) %>%
    mutate(Subset = "noMDD")

all <- rbind(full, mdd, nomdd)

colors <- c(
    "Continuation" = "#50C878",
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF")

all <- all %>%
    mutate(Subset = case_when(
    Subset == "Full" ~ "Full (N=114,891)",
    Subset == "MDD" ~ "MDD (N=57,647)",
    Subset == "noMDD" ~ "No MDD (N=57,244)",
    TRUE ~ NA_character_))

p5 <- ggplot(all, aes(x = Category.treatment, y = n, fill = FinalClassification.treatment)) +
    geom_bar(position = "fill", stat = "identity") +
  scale_y_continuous(labels = scales::percent) +
  ylab("Percentage") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Antidepressant") +
  ylab("% (All of Us)") +
  labs(fill = "Antidepressant\nOutcome") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),  # x, y coordinates (0-1 scale)
    legend.justification = c(1, 1),   # anchor point (right, top)
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold"),
      strip.text = element_text(face = "bold")
  ) +
 facet_wrap(Subset ~ .) +
  scale_y_continuous(
    breaks = seq(0, 70000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))  # 10K, 20K, etc.
p5

In [ ]:
#-- proportions plots for lifelines
library(readxl)
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table6")

dat <- dat %>%
    mutate(Subset = case_when(
    Subset == "Full" ~ "Full (N=13,074)",
    Subset == "MDD" ~ "MDD (N=4,575)",
    Subset == "noMDD" ~ "No MDD (N=8,499)",
    TRUE ~ Subset))

p5b <- ggplot(dat, aes(x = DrugClass.treatment, y = n, fill = FinalClassification.treatment)) +
    geom_bar(position = "fill", stat = "identity") +
  scale_y_continuous(labels = scales::percent) +
  ylab("Percentage") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Antidepressant") +
  ylab("% (Lifelines)") +
  labs(fill = "Antidepressant\nOutcome") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),  # x, y coordinates (0-1 scale)
    legend.justification = c(1, 1),   # anchor point (right, top)
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold"),
      strip.text = element_text(face = "bold")
  ) +
 facet_wrap(Subset ~ .) +
  scale_y_continuous(
    breaks = seq(0, 70000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))  # 10K, 20K, etc.
p5b

In [ ]:
#-- Find distribution of age at first dispense
age <- both %>%
    select(person_id, age_at_first_AD) %>%
    distinct()

# Calculate stats for annotation
mean_age <- mean(age$age_at_first_AD, na.rm = TRUE)
sd_age <- sd(age$age_at_first_AD, na.rm = TRUE)
mean_age
sd_age

p6 <- ggplot(age, aes(x = age_at_first_AD)) +
  geom_density(fill = "#4A90D9", colour = "#2C5F8A", alpha = 0.4, linewidth = 0.8) +
  geom_vline(xintercept = 47.6, linetype = "dashed", colour = "#2C5F8A", linewidth = 0.8) +
  annotate("text", x = 49.6, y = 0.0245, 
           label = "Mean = 47.3 (SD = 15.2)", 
           hjust = 0, colour = "#2C5F8A", size = 4) +
  scale_x_continuous(limits = c(15, 95), breaks = seq(20, 90, 20)) +
  labs(x = "Age at first recorded antidepressant prescription (years)", y = "Density") +
  theme_minimal() +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    axis.line = element_line(colour = "grey50")
  )
p6

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Age_First_AD_Dispense.png",
  plot     = p6,
  width    = 8,
  height   = 6,
  dpi      = 600
)


In [ ]:
sex <- both %>%
    select(person_id, sex_at_birth) %>%
    distinct()
table(sex$sex_at_birth, useNA = "always")

In [ ]:
#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))
anc_df <- anc_df %>% select(research_id, ancestry_pred)
ancestries <- unique(anc_df$ancestry_pred)

dat <- both %>%
    left_join(anc_df, by = c("person_id" = "research_id"))
dat <- dat %>%
          mutate(
            ancestry_pred = factor(ancestry_pred)
      )

In [ ]:
anc <- dat %>%
    select(person_id, ancestry_pred) %>%
    distinct()
table(anc$ancestry_pred, useNA = "always")
650+183+474

In [ ]:
dim(dat)
length(unique(dat$person_id))

In [ ]:
mdd <- both %>%
    select(person_id, Depression) %>%
    distinct()
table(mdd$Depression, useNA = "always")

In [ ]:
#=== Number of unique switches
switches <- final %>%
    filter(FinalClassification.treatment == "Switching") %>%
    group_by(person_id) %>%
    summarise(
        Total = n())
switches <- switches %>%
    mutate(
    Total = case_when(
    Total >= 8 ~ "8-11",
    TRUE ~ as.character(Total)))

no_switches <- final %>%
    filter(!person_id %in% switches$person_id)
num_no_switches <- length(unique(no_switches$person_id))
num_no_switches

total <- switches %>%
    group_by(Total) %>%
    summarise(Count = n())

total$Total <- as.factor(total$Total)

p7 <- ggplot(total, aes(x = Total, y = Count, fill = Total)) +
    geom_col() + 
    geom_text(
      aes(label = scales::comma(Count)),
      vjust = -0.5,
      size = 3
    ) +
    labs(y = "All of Us Participant Count", x = "Total Switches") +
    theme_classic() +
    paletteer::scale_fill_paletteer_d("ggsci::purple_material", direction = -1) +
    scale_y_continuous(
      breaks = seq(0, 25000, by = 5000),
      labels = function(x) paste0(x / 1000, "K"),
      expand = expansion(mult = c(0, 0.1))
    ) +
    theme(
      legend.position = "none",
      axis.title = element_text(face = "bold"),
      axis.text.x = element_text(face = "bold")
    )
p7

In [ ]:
#-- First switch
first_switch <- both %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- First continuation (among non-switchers)
C <- both %>%
  filter(!person_id %in% first_switch$person_id) %>%
  filter(FinalClassification.treatment == "Continuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- Input data for switching vs. continuation
dat <- rbind(first_switch, C)
length(unique(dat$person_id))

dat <- dat %>%
  mutate(
    has_any_AD_modulator_ever = case_when(
      has_AD_cyp2d6_modulator_any_followup == 1L | 
      has_AD_cyp2c19_modulator_any_followup == 1L ~ 1L,
      TRUE ~ 0L
    )
  )

#== Create subset excluding AD modulators
dat_noAD <- dat %>%
    filter(has_any_AD_modulator_ever == 0L)
length(unique(dat_noAD$person_id))

dat_onlyAD <- dat %>%
    filter(has_any_AD_modulator_ever == 1L)

dat_AD <- dat

cyp_noAD <- dat_noAD %>% select(
    person_id, CYP2C19_any_followup_phenoconverted, CYP2C19,
    CYP2D6_any_followup_phenoconverted, CYP2D6, 
    CYP2C19_any_followup_phenoconverted, CYP2C19, 
    CYP2B6_any_followup_phenoconverted, CYP2B6) %>%
    distinct()

cyp_AD <- dat_AD %>% select(
    person_id, CYP2C19_any_followup_phenoconverted, CYP2C19,
    CYP2D6_any_followup_phenoconverted, CYP2D6, 
    CYP2C19_any_followup_phenoconverted, CYP2C19, 
    CYP2B6_any_followup_phenoconverted, CYP2B6) %>%
    distinct()

In [ ]:
#-- % of cohort that have a mismatch for at least one of the three genes
library(dplyr)
library(ComplexUpset)
library(ggplot2)

cyp_noAD_changed <- cyp_noAD %>%
  mutate(
    CYP2C19_changed = case_when(
      !is.na(CYP2C19) & !is.na(CYP2C19_any_followup_phenoconverted) &
        CYP2C19 != CYP2C19_any_followup_phenoconverted ~ 1L,
      !is.na(CYP2C19) & !is.na(CYP2C19_any_followup_phenoconverted) ~ 0L,
      TRUE ~ NA_integer_
    ),
    CYP2D6_changed = case_when(
      !is.na(CYP2D6) & !is.na(CYP2D6_any_followup_phenoconverted) &
        CYP2D6 != CYP2D6_any_followup_phenoconverted ~ 1L,
      !is.na(CYP2D6) & !is.na(CYP2D6_any_followup_phenoconverted) ~ 0L,
      TRUE ~ NA_integer_
    ),
    CYP2B6_changed = case_when(
      !is.na(CYP2B6) & !is.na(CYP2B6_any_followup_phenoconverted) &
        CYP2B6 != CYP2B6_any_followup_phenoconverted ~ 1L,
      !is.na(CYP2B6) & !is.na(CYP2B6_any_followup_phenoconverted) ~ 0L,
      TRUE ~ NA_integer_
    )
  )

table(cyp_noAD_changed$CYP2C19_changed); table(cyp_noAD_changed$CYP2D6_changed); table(cyp_noAD_changed$CYP2B6_changed)

In [ ]:
upset_data <- cyp_noAD_changed %>%
  filter(
    !is.na(CYP2C19_changed) | !is.na(CYP2D6_changed) | !is.na(CYP2B6_changed)
  ) %>%
  mutate(
    CYP2C19_changed = coalesce(CYP2C19_changed, 0L) == 1,
    CYP2D6_changed  = coalesce(CYP2D6_changed, 0L) == 1,
    CYP2B6_changed  = coalesce(CYP2B6_changed, 0L) == 1
  )

upset_plot <- upset(
  upset_data,
  intersect = c("CYP2C19_changed", "CYP2D6_changed", "CYP2B6_changed"),
  name = "Misclassification",
  base_annotations = list(
    "Intersection size" = intersection_size(counts = TRUE)
  )
)

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/CYP_misclassifications_upset.png",
  plot     = upset_plot,
  width    = 9,
  height   = 6,
  dpi      = 600
)


In [ ]:
cyp_noAD_changed <- cyp_noAD_changed %>%
  mutate(
    any_change = case_when(
      (CYP2C19_changed == 1 | CYP2D6_changed == 1 | CYP2B6_changed == 1) &
        (!is.na(CYP2C19_changed) | !is.na(CYP2D6_changed) | !is.na(CYP2B6_changed)) ~ 1L,
      (!is.na(CYP2C19_changed) | !is.na(CYP2D6_changed) | !is.na(CYP2B6_changed)) ~ 0L,
      TRUE ~ NA_integer_
    )
  )
table(cyp_noAD_changed$any_change, useNA = "always")

In [ ]:
library(tidyr)
library(dplyr)
library(stringr)

cyp_summary_noAD <- cyp_noAD %>%
  select(-person_id) %>%
  pivot_longer(everything(), names_to = "Gene", values_to = "Category") %>%
  filter(!is.na(Category)) %>%
  group_by(Gene, Category) %>%
  summarise(Count = n(), .groups = "drop") %>%
  group_by(Gene) %>%
  mutate(Prevalence = Count / sum(Count) * 100) %>%
  mutate(Method = ifelse(str_detect(Gene, "_any_followup_phenoconverted"), "Phenoconversion-adjusted", "Genetic")) %>%
  mutate(Gene = gsub("_any_followup_phenoconverted", "", Gene)) 

cyp_summary_AD <- cyp_AD %>%
  select(-person_id) %>%
  pivot_longer(everything(), names_to = "Gene", values_to = "Category") %>%
  filter(!is.na(Category)) %>%
  group_by(Gene, Category) %>%
  summarise(Count = n(), .groups = "drop") %>%
  group_by(Gene) %>%
  mutate(Prevalence = Count / sum(Count) * 100) %>%
  mutate(Method = ifelse(str_detect(Gene, "_any_followup_phenoconverted"), "Phenoconversion-adjusted", "Genetic")) %>%
  mutate(Gene = gsub("_any_followup_phenoconverted", "", Gene)) 

In [ ]:
#-- change in metabolizer status prevalence
cyp_summary_noAD

In [ ]:
cyp_summary_noAD$Gene = factor(cyp_summary_noAD$Gene, levels = c("CYP2D6", "CYP2C19", "CYP2B6"))
cyp_summary_noAD$Method = factor(cyp_summary_noAD$Method, levels = c("Genetic", "Phenoconversion-adjusted"))
cyp_summary_noAD$Category = factor(cyp_summary_noAD$Category, levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid"))

colors <- c("Genetic" = "#507DBC", "Phenoconversion-adjusted" = "#7A3B69")


p8 <- ggplot(cyp_summary_noAD, aes(x = Category, y = Prevalence, fill = Method)) +
    facet_wrap(~Gene, scale = "free_x") +
    geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.7) + 
    labs(y = "Prevalence (%)", x = "Metabolizer Status", title = "Phenoconversion (excluding antidepressant modulators)") +
    theme_classic() +
    scale_fill_manual(values = colors, name = "Method") +
    theme(
        axis.title = element_text(face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, color = "black"),
        strip.text = element_text(face = "bold"),
        legend.position = c(0.5, 0.95),
        legend.justification = c(0.5, 1),
        legend.direction = "vertical",
        legend.background = element_rect(fill = "white", color = NA),
        legend.title = element_text(face = "bold")
    )
p8

cyp_summary_AD$Gene = factor(cyp_summary_AD$Gene, levels = c("CYP2D6", "CYP2C19", "CYP2B6"))
cyp_summary_AD$Method = factor(cyp_summary_AD$Method, levels = c("Genetic", "Phenoconversion-adjusted"))
cyp_summary_AD$Category = factor(cyp_summary_AD$Category, levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid"))


p9 <- ggplot(cyp_summary_AD, aes(x = Category, y = Prevalence, fill = Method)) +
    facet_wrap(~Gene, scale = "free_x") +
    geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.7) + 
    labs(y = "Prevalence (%)", x = "Metabolizer Status", title = "Phenoconversion with all Modulators") +
    theme_classic() +
    scale_fill_manual(values = colors, name = "Method") +
    theme(
        axis.title = element_text(face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, color = "black"),
        strip.text = element_text(face = "bold"),
        legend.position = c(0.5, 0.95),   # centered horizontally, near top
        legend.justification = c(0.5, 1), # anchor at center-top
        legend.direction = "vertical",
        legend.background = element_rect(fill = "white", color = NA),
        legend.title = element_text(face = "bold")
    )
p9

In [ ]:
library(cowplot)

first <- plot_grid(p1, p7, labels = c("A", "B"), nrow = 1)
first

In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Main_Summary.png",
  plot     = first,
  width    = 10,
  height   = 5,
  dpi      = 600
)


In [ ]:
#-- PGS figure

top <- plot_grid(p2, p2b, nrow = 1, labels = c("A", "B"))
bottom <- plot_grid(p5, p5b, nrow = 1, labels = c("C", "D"))

In [ ]:
all <- plot_grid(top, bottom, nrow =2)
all

In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Class_Switch_Summary.png",
  plot     = all,
  width    = 10,
  height   = 10,
  dpi      = 600
)


In [ ]:
library(readxl)
mod <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_Modulator_Prev_Full")
ad_mod <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_AD_Modulator_Prevalence")

In [ ]:
strength_inhibs <- mod %>%
  filter(Window == "any_followup") %>%
  filter(Gene != "ALL_COMBINED") %>%
  select(Gene, 
         Pct_Strong_Inhibitors, Pct_Moderate_Inhibitors, Pct_Weak_Inhibitors) %>%
  pivot_longer(
    cols = starts_with("Pct_"),
    names_to = "Strength",
    values_to = "Prevalence"
  ) %>%
  filter(!is.na(Prevalence)) %>%
  mutate(
    Strength = gsub("Pct_|_Inhibitors", "", Strength),
    Strength = factor(Strength, levels = c("Strong", "Moderate", "Weak")),
    Direction = "Non-AD Inhibitor")


strength_ind <- mod %>%
  filter(Window == "any_followup") %>%
  select(Gene, 
        Pct_Moderate_Inducers, Pct_Weak_Inducers) %>%
  pivot_longer(
    cols = starts_with("Pct_"),
    names_to = "Strength",
    values_to = "Prevalence"
  ) %>%
  filter(!is.na(Prevalence)) %>%
  mutate(
    Strength = gsub("Pct_|_Inducers", "", Strength),
    Strength = factor(Strength, levels = c("Moderate", "Weak")),
    Direction = "Non-AD Inducer"
  )

strength_ad_inhibs <- ad_mod %>%
  filter(Window == "any_followup") %>%
  filter(Gene != "ALL_COMBINED") %>%
  select(Gene, 
         Pct_Strong_Inhibitors, Pct_Moderate_Inhibitors, Pct_Weak_Inhibitors) %>%
  pivot_longer(
    cols = starts_with("Pct_"),
    names_to = "Strength",
    values_to = "Prevalence"
  ) %>%
  filter(!is.na(Prevalence)) %>%
  mutate(
    Strength = gsub("Pct_|_Inhibitors", "", Strength),
    Strength = factor(Strength, levels = c("Strong", "Moderate", "Weak")),
    Direction = "AD Inhibitor")

strength <- rbind(strength_inhibs, strength_ind, strength_ad_inhibs)
strength$Gene = factor(strength$Gene, levels = c("CYP2D6", "CYP2C19", "CYP2B6"))
strength$Direction = factor(strength$Direction, levels = c("Non-AD Inducer", "Non-AD Inhibitor", "AD Inhibitor"))

colors <- c("Strong" = "#EF476F", "Moderate" = "#FFD166", "Weak" = "#26547C")

p10 <- ggplot(strength, aes(x = Direction, y = Prevalence, fill = Strength)) +
  geom_col(position = "stack", width = 0.9) +
  facet_wrap(~ Gene, ncol = 3) +
  scale_fill_manual(values = colors, name = "Strength") +
  labs(
    x = NULL,
    y = "Prevalence (%)",
    fill = "Strength",
    title = "CYP Modulators"
  ) +
  theme_classic() +
  theme(
      axis.text = element_text(color = "black"),
      axis.text.x = element_text(angle = 60, hjust = 1),
      axis.title = element_text(face = "bold"),
      strip.text = element_text(face = "bold"),
        legend.position = c(0.85, 0.95),   # centered horizontally, near top
        legend.justification = c(0.85, 1), # anchor at center-top
        legend.direction = "vertical",
        legend.background = element_rect(fill = "white", color = NA),
        legend.title = element_text(face = "bold"))
p10

In [ ]:
b <- plot_grid(p10, p8, nrow = 2, labels = c("A", "B"), rel_heights = c(1, 1))
b

In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Phenoconversion_Summary.png",
  plot     = b,
  width    = 8,
  height   = 8,
  dpi      = 600
)

In [ ]:
library(readxl)
mod <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_Modulator_Prev")

In [ ]:

strength_inhibs <- mod %>%
  filter(Window == "any_followup") %>%
  select(Drug, Gene, 
         Pct_Strong_Inhibitors, Pct_Moderate_Inhibitors, Pct_Weak_Inhibitors) %>%
  pivot_longer(
    cols = starts_with("Pct_"),
    names_to = "Strength",
    values_to = "Prevalence"
  ) %>%
  filter(!is.na(Prevalence)) %>%
  mutate(
    Strength = gsub("Pct_|_Inhibitors", "", Strength),
    Strength = factor(Strength, levels = c("Strong", "Moderate", "Weak")),
    Direction = "Inhibitor")


strength_ind <- mod %>%
  filter(Window == "any_followup") %>%
  select(Drug, Gene, 
        Pct_Moderate_Inducers, Pct_Weak_Inducers) %>%
  pivot_longer(
    cols = starts_with("Pct_"),
    names_to = "Strength",
    values_to = "Prevalence"
  ) %>%
  filter(!is.na(Prevalence)) %>%
  mutate(
    Strength = gsub("Pct_|_Inducers", "", Strength),
    Strength = factor(Strength, levels = c("Moderate", "Weak")),
    Direction = "Inducer"
  )

strength <- rbind(strength_inhibs, strength_ind)

strength$Gene = factor(strength$Gene, levels = c("CYP2D6", "CYP2C19", "CYP2B6"))
colors <- c("Strong" = "#EF476F", "Moderate" = "#FFD166", "Weak" = "#26547C")

p <- ggplot(strength, aes(x = Drug, y = Prevalence, fill = Strength)) +
  geom_col(position = "stack") +
  facet_wrap(Gene~ Direction, ncol = 5) +
  coord_flip() +
  labs(
    x = NULL,
    y = "Prevalence (%)",
    fill = "Inhibitor Strength"
  ) +
  scale_fill_manual(values = colors, name = "Strength") +
  theme_classic() +
  theme(
      axis.text = element_text(color = "black"),
      axis.title = element_text(face = "bold"),
      strip.text = element_text(face = "bold"),
        legend.position = "bottom")


In [ ]:
p
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_DrugSpecific_Modulator_Prev.png",
  plot     = p,
  width    = 8,
  height   = 9,
  dpi      = 600
)

In [ ]:
#-- Frequency of genetically-inferred CYP phenotypes
cyp <- final %>%
    select(person_id, CYP2D6, CYP2C19, CYP2B6) %>%
    distinct() %>%
    pivot_longer(cols = c(CYP2D6, CYP2C19, CYP2B6),
                 names_to = "Gene",
                 values_to = "Phenotype")

group_counts <- cyp %>%
    group_by(Gene, Phenotype) %>%
    summarise(NumPeople = n(), .groups = "drop") %>%
    group_by(Gene) %>%
    mutate(Pct = NumPeople / sum(NumPeople[!is.na(Phenotype)]) * 100)
group_counts <- group_counts %>%
    mutate(
        Pct = case_when(
        is.na(Phenotype) ~ NA_integer_, 
        TRUE ~ Pct)) %>%
    mutate(
        Ancestry = "All") %>%
    select(Ancestry, Gene, Phenotype, NumPeople, Pct)

In [ ]:
cyp_anc <- cyp %>%
    left_join(anc_df, by = c("person_id" = "research_id"))
cyp_anc <- cyp_anc %>%
    mutate(
        Ancestry = case_when(
        ancestry_pred == "afr" ~ "African",
        ancestry_pred == "amr" ~ "American",
        ancestry_pred == "eas" ~ "East Asian",
        ancestry_pred == "sas" ~ "South Asian",
        ancestry_pred == "mid" ~ "Middle Eastern",
        ancestry_pred == "eur" ~ "European",
        TRUE ~ NA_character_))

group_counts_anc <- cyp_anc %>%
    group_by(Ancestry, Gene, Phenotype) %>%
    summarise(NumPeople = n(), .groups = "drop") %>%
    group_by(Ancestry, Gene) %>%
    mutate(Pct = NumPeople / sum(NumPeople[!is.na(Phenotype)]) * 100) %>%
    mutate(
        Pct = case_when(
        is.na(Phenotype) ~ NA_integer_, 
        TRUE ~ Pct))

In [ ]:
all_counts <- rbind(group_counts, group_counts_anc)
all_counts <- all_counts %>%
    filter(!is.na(Ancestry)) %>%
    filter(NumPeople >= 20)
all_counts_no_na <- all_counts %>%
    filter(!is.na(Phenotype))

all_counts_no_na$Gene <- factor(all_counts_no_na$Gene, levels = c("CYP2D6", "CYP2C19", "CYP2B6"))
all_counts_no_na$Phenotype <- factor(all_counts_no_na$Phenotype, levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid"))

ancestry_n <- all_counts_no_na %>%
    group_by(Ancestry) %>%
    summarise(N = sum(NumPeople) / n_distinct(Gene), .groups = "drop") %>%
    arrange(desc(N))  # sort by N

all_counts_no_na <- all_counts_no_na %>%
    left_join(ancestry_n, by = "Ancestry") %>%
    mutate(Ancestry_label = paste0(Ancestry, "\n(N=", scales::comma(N), ")"),
           Ancestry_label = factor(Ancestry_label, levels = unique(Ancestry_label[order(-N)])))

phenotype_colors <- c(
  "Poor"         = "#D4785A",
  "Intermediate" = "#C4C45A",
  "Normal"       = "#4DBF9F",
  "Rapid"        = "#6AAAD4",
  "Ultrarapid"   = "#C47DC4" 
)

pp <- ggplot(all_counts_no_na, aes(x = Ancestry_label, y = Pct, fill = Phenotype)) +
  geom_bar(position = "stack", stat = "identity") +
  facet_wrap(Gene ~ ., scales = "free_x", nrow = 1) +
  theme_classic() +
  xlab("Gene") +
  ylab("Percentage (%)") +
  labs(fill = "Genetic\nClassification", title = NULL) +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.text.y = element_text(color = "black"),
    axis.title = element_text(face = "bold"),
    legend.background = element_rect(fill = "white", color = NA),
    legend.position = "bottom",
    strip.text = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 100, by = 20),
    labels = function(x) paste0(x, "%")) +
  guides(fill = guide_legend(nrow = 2)) +
  scale_fill_manual(values = phenotype_colors, name = "Genetic\nClassification")
pp

In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/CYP_Pheno_Frequency.png",
  plot     = pp,
  width    = 9,
  height   = 6,
  dpi      = 600
)

In [ ]:
library(openxlsx)
wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "CYP_Phenotype_Frequency")
addWorksheet(wb, "CYP_Phenotype_Frequency")
writeData(wb, "CYP_Phenotype_Frequency", all_counts)

saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)
